In [0]:
import logging
import uuid
import random
from pyspark.sql import Row

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

RAW_MARKET_PATH = "abfss://raw@cryptodl.dfs.core.windows.net/CSV_Streaming_Data_Source_3/"
S1_PATH         = "abfss://bronzelayer@cryptodl.dfs.core.windows.net/Batch_Data_Source_1/"

log.info("Starting Source 3 — Market Metrics Generation")
log.info(f"Target path: {RAW_MARKET_PATH}")

log.info("Reading S1 combos from Bronze...")
s1 = spark.read.parquet(S1_PATH)


sampled = (
    s1.select("Symbol", "trade_date")
      .distinct()
      .orderBy("Symbol", "trade_date")
      .limit(50)
      .collect()
)
log.info(f"Sampled: {len(sampled)} combos — deterministic, same every run")

events = []
for row in sampled:
    symbol   = row["Symbol"]
    date_str = str(row["trade_date"])[:10]
    events.append(
        Row(
            metric_id         = str(uuid.uuid4()),
            symbol            = symbol,
            trade_date        = date_str,
            active_addresses  = random.randint(10000, 2000000),
            transaction_count = random.randint(5000, 500000),
            network_fees      = round(random.uniform(0.1, 15.0), 2),
            exchange_inflow   = random.randint(1000, 100000),
            exchange_outflow  = random.randint(1000, 100000),
            event_timestamp   = f"{date_str}T{random.randint(0,23):02d}:{random.randint(0,59):02d}:00"
        )
    )

log.info(f"Generated {len(events)} market metric events")
log.info(f"Sample — symbol: {events[0].symbol} | date: {events[0].trade_date}")

log.info("Writing to Bronze (append mode)...")
(
    spark.createDataFrame(events)
         .coalesce(1)
         .write
         .mode("append")
         .option("header", "true")
         .csv(RAW_MARKET_PATH)
)

log.info(f"Done! {len(events)} market metric events written to Bronze")
log.info("=" * 50)